# Solution A: Final Development Model for NLI

## 1. Import libraries

In [1]:
import re
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load training & development data

In [2]:
train_df = pd.read_csv("../data/raw/train.csv")
dev_df = pd.read_csv("../data/raw/dev.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
display(train_df.head())

Train shape: (24432, 3)
Dev shape: (6736, 3)


,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1
3,"""How d'you know he'll sign me on?""Anse studie...",Anse looked at himself in a cracked mirror.,1
4,In the light of the candles his cheeks looked ...,Drew regarded his best friend and noted that i...,1


## 3. Build shared TF-IDF representations

In [3]:
shared_vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

shared_train_corpus = pd.concat([
    train_df["premise"].astype(str),
    train_df["hypothesis"].astype(str)
], axis=0)

shared_vectorizer.fit(shared_train_corpus)

X_train_p = shared_vectorizer.transform(train_df["premise"].astype(str))
X_train_h = shared_vectorizer.transform(train_df["hypothesis"].astype(str))

X_dev_p = shared_vectorizer.transform(dev_df["premise"].astype(str))
X_dev_h = shared_vectorizer.transform(dev_df["hypothesis"].astype(str))

print("Premise TF-IDF shape:", X_train_p.shape)
print("Hypothesis TF-IDF shape:", X_train_h.shape)

Premise TF-IDF shape: (24432, 30000)
Hypothesis TF-IDF shape: (24432, 30000)


## 4. Construct vector interaction features

In [4]:
X_train_absdiff = abs(X_train_p - X_train_h)
X_dev_absdiff = abs(X_dev_p - X_dev_h)

X_train_prod = X_train_p.multiply(X_train_h)
X_dev_prod = X_dev_p.multiply(X_dev_h)

## 5. Define helper functions for lexical feature extraction

In [5]:
NEG_WORDS = {"not", "no", "never", "none", "nothing"}

def tokenize(text):
    text = str(text).lower()
    return re.findall(r"\b\w+\b|n't", text)

def token_set(text):
    return set(tokenize(text))

def jaccard(row):
    p = token_set(row["premise"])
    h = token_set(row["hypothesis"])
    union = p | h
    if not union:
        return 0.0
    return len(p & h) / len(union)

def shared_count(row):
    p = token_set(row["premise"])
    h = token_set(row["hypothesis"])
    return len(p & h)

def premise_covers_hypothesis(row):
    p = token_set(row["premise"])
    h = token_set(row["hypothesis"])
    if len(h) == 0:
        return 0.0
    return len(p & h) / len(h)

def hypothesis_covers_premise(row):
    p = token_set(row["premise"])
    h = token_set(row["hypothesis"])
    if len(p) == 0:
        return 0.0
    return len(p & h) / len(p)

def neg_count(text):
    tokens = tokenize(text)
    return sum(1 for t in tokens if t in NEG_WORDS or t == "n't")

## 6. Construct handcrafted pairwise features

In [6]:
train_cos = cosine_similarity(X_train_p, X_train_h).diagonal()
dev_cos = cosine_similarity(X_dev_p, X_dev_h).diagonal()

train_extra = pd.DataFrame({
    "premise_len": train_df["premise"].astype(str).str.len(),
    "hypothesis_len": train_df["hypothesis"].astype(str).str.len(),
    "len_diff": (
        train_df["premise"].astype(str).str.len()
        - train_df["hypothesis"].astype(str).str.len()
    ).abs(),
    "jaccard": train_df.apply(jaccard, axis=1),
    "shared_count": train_df.apply(shared_count, axis=1),
    "premise_covers_hypothesis": train_df.apply(premise_covers_hypothesis, axis=1),
    "hypothesis_covers_premise": train_df.apply(hypothesis_covers_premise, axis=1),
    "premise_neg": train_df["premise"].astype(str).apply(neg_count),
    "hypothesis_neg": train_df["hypothesis"].astype(str).apply(neg_count),
    "tfidf_cosine": train_cos,
})

train_extra["neg_diff"] = (train_extra["premise_neg"] - train_extra["hypothesis_neg"]).abs()

dev_extra = pd.DataFrame({
    "premise_len": dev_df["premise"].astype(str).str.len(),
    "hypothesis_len": dev_df["hypothesis"].astype(str).str.len(),
    "len_diff": (
        dev_df["premise"].astype(str).str.len()
        - dev_df["hypothesis"].astype(str).str.len()
    ).abs(),
    "jaccard": dev_df.apply(jaccard, axis=1),
    "shared_count": dev_df.apply(shared_count, axis=1),
    "premise_covers_hypothesis": dev_df.apply(premise_covers_hypothesis, axis=1),
    "hypothesis_covers_premise": dev_df.apply(hypothesis_covers_premise, axis=1),
    "premise_neg": dev_df["premise"].astype(str).apply(neg_count),
    "hypothesis_neg": dev_df["hypothesis"].astype(str).apply(neg_count),
    "tfidf_cosine": dev_cos,
})

dev_extra["neg_diff"] = (dev_extra["premise_neg"] - dev_extra["hypothesis_neg"]).abs()

display(train_extra.head())

,premise_len,hypothesis_len,len_diff,jaccard,shared_count,premise_covers_hypothesis,hypothesis_covers_premise,premise_neg,hypothesis_neg,tfidf_cosine,neg_diff
0,53,113,60,0.280000,7,0.333333,0.636364,0,1,0.302948,1
1,29,51,22,0.333333,4,0.400000,0.666667,1,1,0.355069,0
2,41,62,21,0.636364,7,0.636364,1.000000,0,0,0.719915,0
3,122,44,78,0.107143,3,0.375000,0.130435,0,0,0.194861,0
4,185,98,87,0.195122,8,0.500000,0.242424,1,0,0.182190,1


## 7. Combine all feature blocks

In [7]:
X_train_extra = csr_matrix(train_extra.values)
X_dev_extra = csr_matrix(dev_extra.values)

X_train = hstack([
    X_train_p,
    X_train_h,
    X_train_absdiff,
    X_train_prod,
    X_train_extra
])

X_dev = hstack([
    X_dev_p,
    X_dev_h,
    X_dev_absdiff,
    X_dev_prod,
    X_dev_extra
])

y_train = train_df["label"].values
y_dev = dev_df["label"].values

print("Final X_train shape:", X_train.shape)
print("Final X_dev shape:", X_dev.shape)

Final X_train shape: (24432, 120011)
Final X_dev shape: (6736, 120011)


## 8. Tune logistic regression on the development set

In [8]:
configs = [
    {"name": "liblinear_C0.10", "C": 0.10, "solver": "liblinear"},
    {"name": "liblinear_C0.15", "C": 0.15, "solver": "liblinear"},
    {"name": "liblinear_C0.20", "C": 0.20, "solver": "liblinear"},
    {"name": "liblinear_C0.25", "C": 0.25, "solver": "liblinear"},
    {"name": "liblinear_C0.30", "C": 0.30, "solver": "liblinear"},
    {"name": "liblinear_C0.40", "C": 0.40, "solver": "liblinear"},
    {"name": "liblinear_C0.50", "C": 0.50, "solver": "liblinear"},
]

In [9]:
results = []

for cfg in configs:
    clf = LogisticRegression(
        max_iter=3000,
        C=cfg["C"],
        solver=cfg["solver"]
    )
    clf.fit(X_train, y_train)
    dev_preds = clf.predict(X_dev)

    acc = accuracy_score(y_dev, dev_preds)
    macro_f1 = f1_score(y_dev, dev_preds, average="macro")
    weighted_f1 = f1_score(y_dev, dev_preds, average="weighted")
    mcc = matthews_corrcoef(y_dev, dev_preds)

    results.append({
        "model_name": cfg["name"],
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "mcc": mcc
    })

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_df

,model_name,accuracy,macro_f1,weighted_f1,mcc
5,liblinear_C0.40,0.691508,0.690503,0.691079,0.381828
6,liblinear_C0.50,0.689578,0.688644,0.689201,0.377965
4,liblinear_C0.30,0.689133,0.687977,0.688597,0.377056
3,liblinear_C0.25,0.688688,0.687467,0.688105,0.376166
2,liblinear_C0.20,0.685422,0.684140,0.684797,0.369608
1,liblinear_C0.15,0.684679,0.683389,0.684049,0.368116
0,liblinear_C0.10,0.682898,0.681438,0.682142,0.364563


## 9. Select the best configuration

In [10]:
best_row = results_df.iloc[0]
best_row

model_name     liblinear_C0.40
accuracy              0.691508
macro_f1              0.690503
weighted_f1           0.691079
mcc                   0.381828
Name: 5, dtype: object

## 10. Save development predictions

In [11]:
best_cfg = [c for c in configs if c["name"] == best_row["model_name"]][0]

best_clf = LogisticRegression(
    max_iter=3000,
    C=best_cfg["C"],
    solver=best_cfg["solver"]
)

best_clf.fit(X_train, y_train)
best_dev_preds = best_clf.predict(X_dev)

pd.DataFrame({"prediction": best_dev_preds}).to_csv(
    "../outputs/predictions/solution_A_final_dev_predictions.csv",
    index=False
)

print("Saved best tuned dev predictions.")
print(best_cfg)

Saved best tuned dev predictions.
{'name': 'liblinear_C0.40', 'C': 0.4, 'solver': 'liblinear'}


## 11. Report final development metrics

In [12]:
acc = accuracy_score(y_dev, best_dev_preds)
macro_f1 = f1_score(y_dev, best_dev_preds, average="macro")
weighted_f1 = f1_score(y_dev, best_dev_preds, average="weighted")
mcc = matthews_corrcoef(y_dev, best_dev_preds)

print("Accuracy:", acc)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)
print("MCC:", mcc)

Accuracy: 0.6915083135391924
Macro F1: 0.6905025047406108
Weighted F1: 0.6910787493647983
MCC: 0.3818282629610386
